This is to ensamble submission

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax

In [ ]:
LOGITS_FILES = {
    # ---- RoBERTa ----
    "roberta_512": "roberta_processed/logits_roberta_processed_noseed_MAXLEN_512 .npy",
    "roberta_256": "roberta_MAXLEN256/logits_roberta_processed_noseed_MAXLEN_256.npy",

    # ---- DeBERTa ----
    "deberta_512": "deberta_processed/logits_deberta_processed_noseed_MAXLEN_512.npy",
    "deberta_256": "deberta_MAXLEN256/logits_deberta_processed_noseed_MAXLEN_256.npy",
}

In [ ]:
EVAL_CSV = "../data/processed/evaluation_processed.csv"
OUT_PREFIX = "../data/submission/ensemble_submission/submission_"

# pesi (puoi modificarli)
WEIGHTS = {
    "roberta_512": 0.30,
    "roberta_256": 0.25,
    "deberta_512": 0.30,
    "deberta_256": 0.15,
}


In [ ]:
#load data 
df_eval = pd.read_csv(EVAL_CSV)
ids = df_eval["Id"].astype(int).values

logits = {}
for name, path in LOGITS_FILES.items():
    logits[name] = np.load(path)
    print(f"{name}: {logits[name].shape}")

N, C = next(iter(logits.values())).shape
assert C == 7

In [ ]:
#Average

avg_logits = np.mean(list(logits.values()), axis=0)
avg_preds = avg_logits.argmax(axis=1)

pd.DataFrame({
    "Id": ids,
    "Predicted": avg_preds
}).to_csv(f"{OUT_PREFIX}avg.csv", index=False)



In [ ]:
#weighted
weighted_logits = np.zeros_like(avg_logits)

for name, w in WEIGHTS.items():
    weighted_logits += w * logits[name]

weighted_preds = weighted_logits.argmax(axis=1)

pd.DataFrame({
    "Id": ids,
    "Predicted": weighted_preds
}).to_csv(f"{OUT_PREFIX}weighted.csv", index=False)



In [ ]:
#temperature 
probs = []
for l in logits.values():
    probs.append(softmax(l / TEMPERATURE, axis=1))

temp_probs = np.mean(probs, axis=0)
temp_preds = temp_probs.argmax(axis=1)

pd.DataFrame({
    "Id": ids,
    "Predicted": temp_preds
}).to_csv(f"{OUT_PREFIX}temperature.csv", index=False)



In [ ]:
def entropy(p):
    return -np.sum(p * np.log(p + 1e-12), axis=1)


base_probs = softmax(weighted_logits, axis=1)


rob_probs = softmax(
    0.5 * logits["roberta_512"] + 0.5 * logits["roberta_256"],
    axis=1
)

H = entropy(base_probs)
threshold = np.percentile(H, 70)  # ~top 30% hard

final_preds = np.where(
    H[:, None] > threshold,
    rob_probs.argmax(axis=1, keepdims=True),
    base_probs.argmax(axis=1, keepdims=True)
).squeeze()

pd.DataFrame({
    "Id": ids,
    "Predicted": final_preds
}).to_csv(f"{OUT_PREFIX}hardcase.csv", index=False)

print(" All submission done ")

In [ ]:
#######


import numpy as np
import pandas as pd
from scipy.special import softmax

LOGITS_FILES = {
    # ---- RoBERTa ----
    "roberta_512": "roberta_processed/logits_roberta_processed_noseed_MAXLEN_512 .npy",
    "roberta_256": "roberta_MAXLEN256/logits_roberta_processed_noseed_MAXLEN_256.npy",

    # ---- DeBERTa ----
    "deberta_512": "deberta_processed/logits_deberta_processed_noseed_MAXLEN_512.npy",
    "deberta_256": "deberta_MAXLEN256/logits_deberta_processed_noseed_MAXLEN_256.npy"
}

df_eval = pd.read_csv("evaluation_processed.csv")
ids = df_eval["Id"].values

logits = {k: np.load(v) for k, v in LOGITS_FILES.items()}
N, C = next(iter(logits.values())).shape
assert C == 7
avg_logits = np.mean(list(logits.values()), axis=0)
avg_preds = avg_logits.argmax(axis=1)

pd.DataFrame({
    "Id": ids,
    "Predicted": avg_preds
}).to_csv("submission_avg.csv", index=False)
WEIGHTS = {
    "roberta_512": 0.30,
    "roberta_256": 0.25,
    "deberta_512": 0.30,
    "deberta_256": 0.15,
}

weighted_logits = sum(
    WEIGHTS[k] * logits[k] for k in WEIGHTS
)

weighted_preds = weighted_logits.argmax(axis=1)

pd.DataFrame({
    "Id": ids,
    "Predicted": weighted_preds
}).to_csv("submission_weighted.csv", index=False)


def entropy(p):
    return -np.sum(p * np.log(p + 1e-12), axis=1)

base_probs = softmax(weighted_logits, axis=1)

rob_logits = 0.5 * logits["roberta_512"] + 0.5 * logits["roberta_256"]
rob_probs  = softmax(rob_logits, axis=1)

H = entropy(base_probs)
threshold = np.percentile(H, 70)  # top 30%

final_preds = np.where(
    H > threshold,
    rob_probs.argmax(axis=1),
    base_probs.argmax(axis=1)
)

pd.DataFrame({
    "Id": ids,
    "Predicted": final_preds
}).to_csv("submission_hardcase.csv", index=False)
